# Modelagem - Síndrome dos Ovários Policísticos (SOP)

Treino e comparação de 6 modelos de classificação para SOP, seguindo o
mesmo padrão do notebook de modelagem do câncer de mama (`02_modelagem_tabular`).
A análise exploratória está no notebook `06b_eda_sop.ipynb`.

In [1]:
# Celula 1 - Imports
import sys, os
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.pcos.train import train_pcos_models
from src.pcos.predict import PCOSPredictor
from src.pcos.dataset import get_pcos_samples

sns.set_theme(style='whitegrid')
print('Imports carregados')

Imports carregados


In [2]:
# Celula 2 - Treinar modelos (salva em models/pcos/)
modelos, resultados, preprocessor, features = train_pcos_models()
print('Treino concluido!')

C:\Users\ricoi\POSTECH\tech-challenge-fase1\.venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


Treino concluido!


In [3]:
# Celula 3 - Tabela de metricas por modelo
df_resultados = pd.DataFrame(resultados).T
df_resultados = df_resultados.round(4) * 100
df_resultados.columns = ['Acuracia', 'AUC', 'Recall', 'Precisao', 'F1']
df_resultados.index.name = 'Modelo'
print('Metricas (%) por modelo:')
df_resultados

Metricas (%) por modelo:


,Acuracia,AUC,Recall,Precisao,F1
Modelo,,,,,
logistic_regression,88.99,95.09,86.11,81.58,83.78
decision_tree,88.99,91.38,80.56,85.29,82.86
random_forest,93.58,95.13,88.89,91.43,90.14
gradient_boosting,89.91,94.94,80.56,87.88,84.06
svm,89.91,93.82,75.00,93.10,83.08
knn,89.91,93.74,72.22,96.30,82.54


In [4]:
# Celula 4 - Grafico comparativo
df_resultados.plot(kind='bar', figsize=(12, 6))
plt.title('Comparativo de Metricas - SOP', fontsize=14, fontweight='bold')
plt.ylabel('Score (%)')
plt.xlabel('Modelo')
plt.ylim(50, 100)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

# Tambem salvo em outputs/pcos/comparativo.png
os.makedirs('outputs/pcos', exist_ok=True)
plt.savefig('outputs/pcos/comparativo_nb.png', dpi=150, bbox_inches='tight')
print('Grafico salvo em outputs/pcos/comparativo_nb.png')

C:\Users\ricoi\AppData\Local\Temp\ipykernel_22032\1160096702.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Grafico salvo em outputs/pcos/comparativo_nb.png


In [5]:
# Celula 5 - Melhor modelo e predicao de exemplo
melhor = df_resultados['F1'].idxmax()
print(f'Melhor modelo por F1: {melhor} ({df_resultados.loc[melhor, "F1"]:.2f}%)')

# Exemplo: primeira paciente com SOP do dataset
samples = get_pcos_samples()
amostra = samples['samples']['positive']['features']
predictor = PCOSPredictor()
result = predictor.predict(amostra, melhor)
print('\nPredicao de exemplo (paciente real com SOP):')
for k, v in result.items():
    if k != 'features':
        print(f'  {k}: {v}')

Melhor modelo por F1: random_forest (90.14%)



Predicao de exemplo (paciente real com SOP):
  prediction: PCOS
  probability_positive: 0.9043181818181819
  probability_negative: 0.09568181818181809
  model_used: random_forest


In [6]:
# Celula 6 - Feature importance (Random Forest)
rf = modelos['random_forest']
importances = rf.feature_importances_
indices = np.argsort(importances)[::-1]

plt.figure(figsize=(10, 8))
plt.barh(range(15), importances[indices[:15]][::-1], color='steelblue')
plt.yticks(range(15), [features[i] for i in indices[:15]][::-1])
plt.title('Feature Importance - Random Forest (Top 15)', fontsize=14, fontweight='bold')
plt.xlabel('Importancia')
plt.tight_layout()
plt.show()

print('Top 10 features mais importantes:')
for i in range(10):
    print(f'  {i+1}. {features[indices[i]]}: {importances[indices[i]]:.4f}')

Top 10 features mais importantes:
  1. Follicle No. (R): 0.1789
  2. Follicle No. (L): 0.1252
  3. hair growth(Y/N): 0.0571
  4. Skin darkening (Y/N): 0.0528
  5. Weight gain(Y/N): 0.0491
  6. AMH(ng/mL): 0.0363
  7. Cycle(R/I): 0.0269
  8. Cycle length(days): 0.0267
  9. Fast food (Y/N): 0.0246
  10. BMI: 0.0230


C:\Users\ricoi\AppData\Local\Temp\ipykernel_22032\360340157.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [7]:
# Celula 7 - Explicabilidade SHAP (modelo de referencia, como no notebook FIAP)
try:
    import shap
    from src.pcos.dataset import load_pcos_data

    modelo_referencia = modelos["logistic_regression"]
    X_train_raw, X_test_raw, y_train_raw, y_test_raw, _ = load_pcos_data()

    # Pre-processa da mesma forma que o treino (imputacao + escala)
    X_train_processed = preprocessor.transform(X_train_raw)
    X_test_processed = preprocessor.transform(X_test_raw)

    background = X_train_processed[:100]
    explainer = shap.Explainer(modelo_referencia, background, feature_names=features)
    shap_values = explainer(X_test_processed[:50])

    plt.figure(figsize=(12, 8))
    shap.summary_plot(shap_values, X_test_processed[:50], feature_names=features, show=False)
    plt.tight_layout()
    plt.show()
    print("SHAP executado com sucesso para o modelo de referencia: logistic_regression")
except Exception as e:
    print(f"SHAP nao pode ser executado neste ambiente: {e}")


C:\Users\ricoi\POSTECH\tech-challenge-fase1\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


SHAP executado com sucesso para o modelo de referencia: logistic_regression


C:\Users\ricoi\AppData\Local\Temp\ipykernel_22032\1865481530.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
